In [16]:
require_relative 'draw_dot'

false

In [2]:
require 'set'

class Value
    attr_reader :data, :children, :op
    attr_accessor :label, :grad, :backward

    def initialize(data, children = [], operator = '', label: '')
        @data = data
        @grad = 0.0
        @backward = -> {}
        @children = children.to_set # consider other data structures?
        @op = operator
        @label = label
    end

    def inspect
        # similar to python just for the lulz
        "Value:(data=#{@data})"
    end

    def +(operand)
        out = Value.new( self.data + operand.data, [self, operand], '+')
        out.backward = lambda {
            self.grad += 1.0 * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
            operand.grad += 1.0 *out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
        }
        out
    end

    
    def *(operand)
        out = Value.new( self.data * operand.data, [self, operand], '*')
        out.backward = lambda {
            self.grad += operand.data * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
            operand.grad += self.data * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
        }
        out
    end

    # activation (squashing) function
    def tanh
        x = self.data
        t = (Math.exp(2 * x) - 1) / (Math.exp(2 * x) + 1)
        out = Value.new(t, [self], 'tanh - squashing/activation func')

        out.backward = lambda {
            self.grad = (1 - out.data**2) * out.grad
        }
        out
    end

    
    def propagate_all
      topo = []
      visited = Set.new

      build_topo = lambda do |v|
        next if visited.include?(v)

        visited.add(v)
        v.children.each { |child| build_topo.call(child) }
        topo << v
      end

      build_topo.call(self)

      @grad = 1.0
      topo.reverse_each { |v| v.backward.call }
    end
    
end


:propagate_all

In [4]:
# inputs x
x1 = Value.new(2.0, label: 'axon signal from neuron x1')
x2 = Value.new(0.0, label: 'axon signal from neuron  x2')
# weights w
w1 = Value.new(-3.0, label: 'synapse wight w1')
w2 = Value.new(1.0, label: 'synapse wight w2')
# bias of neuron
b = Value.new(6.8813, label: 'Neuron Bias')

# input signals multiplied by the weights of the dendrite they were received
x1w1 = x1*w1; x1w1.label = 'dendrite signal x1w1'
x2w2 = x2*w2; x2w2.label = 'dendrite signal x2w2'

# then they cumulated
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'Sum x1w1x2w2'
# then bias is added to get the BODY OF NEURON, without activation func (output axon)
n = x1w1x2w2 + b; n.label = 'NEURON'
o = n.tanh; o.label = 'output O'

"output O"

false

In [17]:
# auto_graph_o = draw_dot_graphviz(o)
# auto_graph_o.output(svg: "neuron222.svg")